# Text Analytics: History, Workflow, and Practice Areas

Text analytics turns unstructured text into searchable, structured, and decision-ready information.
This notebook connects the historical roots of the field to a practical workflow and to the seven
practice areas that organise most text-mining projects.

It is a companion to `09_nlp_basics`, which focuses on text classification. Here the emphasis is on
the **whole process**: framing the problem, preprocessing, representing text, choosing the right
practice area, summarising, and evaluating the result.

## Learning objectives

By the end of this notebook you can:

- describe the historical roots of text analytics in information retrieval, extraction, and
  summarisation;
- name and explain the seven practice areas of text analytics;
- choose a practice area from a business question;
- run a small end-to-end text workflow with TF-IDF;
- implement extractive summarisation with sentence scoring;
- explain syntax versus semantics and the generalized vector-space model.

## 1. What is text analytics?

Text analytics (also called text mining) is the process of deriving useful patterns, facts, and
relationships from unstructured text. It combines ideas from:

- **information retrieval (IR):** search, ranking, and indexing;
- **information extraction (IE):** turning text into structured facts;
- **computational linguistics and NLP:** syntax, entities, and meaning;
- **statistics and machine learning:** classification, clustering, and topic models;
- **data mining and databases:** scalable storage and analysis;
- **web mining:** links, scale, and networked content.

Text analytics is application-oriented. NLP is often the enabling technology. A text-analytics
project may use NLP tools, but its success is measured by whether it answers a real question.

### A short history

| Period | Development |
| --- | --- |
| 1950s-1960s | Automatic indexing, information retrieval, early automatic summarisation (Luhn) |
| 1970s-1980s | Information extraction, expert systems, computational linguistics |
| 1990s | Web search, vector-space model, TF-IDF, statistical NLP, large text collections |
| 2000s | Text mining, topic models such as LDA, sentiment analysis, big-data infrastructure |
| 2010s | Word embeddings, neural NLP, transformers, pre-trained language models |
| 2020s | Large language models, retrieval-augmented generation, responsible AI |

The modern stack is powerful, but the same questions remain: what is the unit of analysis, what
counts as evidence, how will the output be evaluated, and who will act on it?

## 2. Syntax versus semantics

- **Syntax** is structure: tokens, sentences, part-of-speech tags, phrases, dependencies.
- **Semantics** is meaning: entities, intent, sentiment, topics, and relationships.

A bag-of-words model uses very little syntax. It can still classify topics well, but it may miss
negation ("not good"), word order, and context. Transformer models capture more context, but they
are not automatically correct or explainable.

The **generalized vector-space model** represents each document as a vector of term weights. Terms
are dimensions, documents are points, and cosine similarity measures orientation rather than
length. TF-IDF is the classic weighting scheme:

$$
\text{tfidf}(t,d) = \text{tf}(t,d)\times\log\left(\frac{N}{\text{df}(t)}\right).
$$

- **Term frequency (TF)** rewards terms that occur in a document.
- **Inverse document frequency (IDF)** downweights terms that occur everywhere.
- **Cosine similarity** compares documents or a query and a document.

## 3. The seven practice areas

The seven practice areas are a practical map of text analytics:

| # | Practice area | Question it answers | Typical methods |
| --- | --- | --- | --- |
| 1 | Search and information retrieval | Which documents contain what I need? | Inverted index, TF-IDF, BM25, ranking |
| 2 | Document clustering | What groups exist without labels? | k-means, hierarchical clustering, GMM |
| 3 | Document classification | Which label belongs to this document? | Naive Bayes, logistic regression, SVM |
| 4 | Web mining | What can links and web structure tell us? | Crawling, link analysis, PageRank, social graphs |
| 5 | Information extraction | What facts and relationships are stated? | Regex, rules, NER, relation extraction |
| 6 | Natural language processing | What is the linguistic structure? | Tokenisation, POS tagging, parsing, embeddings |
| 7 | Concept extraction | Which terms and phrases mean similar things? | TF-IDF, co-occurrence, embeddings, taxonomy |

These areas overlap. A single project may search documents, classify them, extract entities, and
summarise the result.

### Choosing a practice area

- Need a specific document? Start with **search/IR**.
- Need groups but no labels? Start with **clustering**.
- Have labels? Start with **classification**.
- Need structured facts? Start with **information extraction**.
- Need linguistic structure? Start with **NLP**.
- Need related terms or a taxonomy? Start with **concept extraction**.
- Need to exploit links or web-scale content? Start with **web mining**.

In [1]:
import os
import warnings

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
warnings.filterwarnings("ignore", message="A NumPy version")
warnings.filterwarnings("ignore", message="Could not find the number of physical cores")
warnings.filterwarnings("ignore", category=UserWarning, module="joblib")

import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, adjusted_rand_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

## 4. A small support-ticket corpus

We use 18 short support tickets across three categories. The text is synthetic, so the example is
safe to share and reproduces without downloads.

In [2]:
documents = [
    ("billing", "I was charged twice for my subscription and need a refund. The invoice shows two payments."),
    ("billing", "Please update my credit card and send a receipt. The invoice total is wrong."),
    ("billing", "My refund has not arrived. The payment was taken twice and the charge is still pending."),
    ("billing", "Can I change my billing cycle? I want a new invoice and a refund for the difference."),
    ("billing", "The coupon did not apply and I paid full price. Please adjust the invoice and refund the charge."),
    ("billing", "I need to cancel my subscription and stop future charges. Please confirm the refund."),
    ("technical", "The app crashes when I upload a photo. I restarted the phone after the update."),
    ("technical", "I cannot log in. The password reset message never arrives after the update."),
    ("technical", "The dashboard is slow and the report page shows an error after the update."),
    ("technical", "Syncing fails with an error message. The app stays on waiting after the update."),
    ("technical", "The export button downloads an empty file. The app shows an error."),
    ("technical", "Two-factor authentication codes are rejected. I cannot log in to the app."),
    ("account", "Please change the username on my profile and update my workspace preferences."),
    ("account", "I want to add a team member and give them admin permissions in the workspace."),
    ("account", "How do I delete my profile and remove my personal data from the workspace?"),
    ("account", "I need to update my company name and address in the profile settings."),
    ("account", "Can I merge two profiles and share one workspace with my team?"),
    ("account", "Please enable single sign-on for our team and configure the workspace domain."),
]

df = pd.DataFrame(documents, columns=["category", "text"])
print("Documents:", len(df))
print(df["category"].value_counts().to_string())
df.head()

Documents: 18
category
billing      6
technical    6
account      6


,category,text
0,billing,I was charged twice for my subscription and ne...
1,billing,Please update my credit card and send a receip...
2,billing,My refund has not arrived. The payment was tak...
3,billing,Can I change my billing cycle? I want a new in...
4,billing,The coupon did not apply and I paid full price...


## 5. Preprocess and represent

A typical preprocessing sequence is:

1. normalise case and whitespace;
2. remove or retain punctuation depending on the task;
3. tokenise;
4. remove stop words;
5. optionally stem or lemmatise;
6. build n-grams or term weights.

For this example we keep punctuation removal and stop-word removal simple, then use TF-IDF with
unigrams and bigrams.

In [3]:
def basic_tokens(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()


sample_tokens = basic_tokens(df.loc[0, "text"])
print("Sample tokens:", sample_tokens[:15])
print("Token count:", len(sample_tokens))

Sample tokens: ['i', 'was', 'charged', 'twice', 'for', 'my', 'subscription', 'and', 'need', 'a', 'refund', 'the', 'invoice', 'shows', 'two']
Token count: 16


In [4]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)

X = vectorizer.fit_transform(df["text"])
terms = vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", X.shape)
print("Number of terms:", len(terms))
print("First terms:", terms[:12])

TF-IDF matrix shape: (18, 195)
Number of terms: 195
First terms: ['add' 'add team' 'address' 'address profile' 'adjust' 'adjust invoice'
 'admin' 'admin permissions' 'app' 'app crashes' 'app shows' 'app stays']


## 6. Practice area 1: search and information retrieval

A query is represented in the same vector space as the documents. Cosine similarity ranks documents
by how closely their term profiles match the query.

In [5]:
query = "refund for a duplicate charge"
query_vector = vectorizer.transform([query])
similarities = cosine_similarity(query_vector, X).ravel()

ranking = np.argsort(similarities)[::-1][:5]
search_results = df.iloc[ranking].copy()
search_results["similarity"] = similarities[ranking]

search_results[["category", "similarity", "text"]].round(3)

,category,similarity,text
2,billing,0.317,My refund has not arrived. The payment was tak...
4,billing,0.279,The coupon did not apply and I paid full price...
0,billing,0.112,I was charged twice for my subscription and ne...
3,billing,0.110,Can I change my billing cycle? I want a new in...
5,billing,0.107,I need to cancel my subscription and stop futu...


This is the core of keyword search. Production search engines add an inverted index, field boosts,
phrase matching, spelling correction, learning-to-rank, and click feedback. The vector-space idea
remains a useful baseline.

## 7. Practice area 2: document clustering

Clustering groups documents without labels. Here we use k-means on TF-IDF vectors and compare the
clusters with the known categories using the adjusted Rand index (ARI).

In [6]:
kmeans = KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE)
cluster_labels = kmeans.fit_predict(X)

print("Adjusted Rand index vs categories:", round(adjusted_rand_score(df["category"], cluster_labels), 3))

cluster_table = pd.crosstab(df["category"], cluster_labels, rownames=["category"], colnames=["cluster"])
cluster_table

Adjusted Rand index vs categories: 1.0


cluster,0,1,2
category,,,
account,0,6,0
billing,6,0,0
technical,0,0,6


The cross-tab shows how the discovered clusters align with the known labels. Clustering is not
classification: it does not use the labels, and the cluster numbers are arbitrary.

## 8. Practice area 3: document classification

Classification uses labelled examples. We split the tickets, fit a TF-IDF + logistic-regression
pipeline, and inspect the test predictions.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["category"],
    test_size=0.33,
    stratify=df["category"],
    random_state=RANDOM_STATE,
)

classifier = LogisticRegression(max_iter=1000)
train_matrix = vectorizer.fit_transform(X_train)
test_matrix = vectorizer.transform(X_test)
classifier.fit(train_matrix, y_train)
predictions = classifier.predict(test_matrix)

print("Test accuracy:", round(accuracy_score(y_test, predictions), 3))
print()
print(classification_report(y_test, predictions, zero_division=0))

Test accuracy: 1.0

              precision    recall  f1-score   support

     account       1.00      1.00      1.00         2
     billing       1.00      1.00      1.00         2
   technical       1.00      1.00      1.00         2

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



For a real project, split before fitting the vectorizer (or put both steps in a pipeline) to avoid
leakage. Here the split is small and illustrative; the important point is the workflow: text ->
representation -> model -> evaluation.

## 9. Practice area 4: web mining

Web mining exploits links, scale, and networked content. A tiny link graph is enough to show the
idea. PageRank-style scores reward pages that are linked to by important pages, not just by many
pages.

In [8]:
pages = ["/home", "/pricing", "/blog/refund", "/docs/api", "/support"]
links = [
    ("/home", "/pricing"),
    ("/home", "/docs/api"),
    ("/pricing", "/support"),
    ("/blog/refund", "/pricing"),
    ("/blog/refund", "/home"),
    ("/support", "/docs/api"),
    ("/docs/api", "/home"),
]

out_degree = Counter(source for source, _ in links)
scores = pd.Series(1 / len(pages), index=pages)

for _ in range(20):
    updated = pd.Series(0.15 / len(pages), index=pages)
    for source, target in links:
        updated[target] += 0.85 * scores[source] / out_degree[source]
    scores = updated

page_table = pd.DataFrame(
    {
        "in_degree": [sum(1 for _, target in links if target == page) for page in pages],
        "pagerank_score": scores.values,
    },
    index=pages,
).sort_values("pagerank_score", ascending=False)

page_table.round(4)

,in_degree,pagerank_score
/docs/api,2,0.3115
/home,2,0.3076
/support,1,0.1774
/pricing,2,0.1735
/blog/refund,0,0.0300


Real web mining also covers crawling, boilerplate removal, duplicate detection, social graphs, and
large-scale link analytics. Always check robots.txt, terms of service, privacy, and copyright.

## 10. Practice area 5: information extraction

Information extraction turns unstructured text into structured facts. Regular expressions handle
well-defined patterns; named-entity recognition and relation extraction handle fuzzier cases.

In [9]:
ticket_text = (
    "Order ORD-1042 for $89.99 was charged on 2026-03-14. "
    "Contact jane.doe@example.com or +1-555-0100 for the refund."
)

patterns = {
    "order_id": r"ORD-\d{4}",
    "amount": r"\$\d+(?:\.\d{2})?",
    "date": r"\d{4}-\d{2}-\d{2}",
    "email": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone": r"\+?\d{1,3}-\d{3}-\d{4}",
}

extracted = {name: re.findall(pattern, ticket_text) for name, pattern in patterns.items()}
extracted

{'order_id': ['ORD-1042'],
 'amount': ['$89.99'],
 'date': ['2026-03-14'],
 'email': ['jane.doe@example.com'],
 'phone': ['+1-555-0100']}

The output is now a structured record that can feed a database, a dashboard, or an alert. The same
idea scales to invoices, contracts, clinical notes, and log files, but requires careful validation.

## 11. Practice area 6: natural language processing

NLP supplies the linguistic layer: tokenisation, sentence splitting, part-of-speech tagging,
lemmatisation, parsing, and entity recognition. This notebook uses lightweight tokenisation; a
production pipeline would normally use a dedicated NLP library.

In [10]:
stop_words = set(TfidfVectorizer(stop_words="english").get_stop_words())
tokens = basic_tokens(df.loc[1, "text"])
content_tokens = [token for token in tokens if token not in stop_words]

print("Original tokens:", tokens)
print()
print("Content tokens:", content_tokens)
print()
print("Most common content tokens:", Counter(content_tokens).most_common(5))

Original tokens: ['please', 'update', 'my', 'credit', 'card', 'and', 'send', 'a', 'receipt', 'the', 'invoice', 'total', 'is', 'wrong']

Content tokens: ['update', 'credit', 'card', 'send', 'receipt', 'invoice', 'total', 'wrong']

Most common content tokens: [('update', 1), ('credit', 1), ('card', 1), ('send', 1), ('receipt', 1)]


Syntax and semantics are different levels of analysis. A tokeniser can tell you that "not" occurs
before "good"; a semantic model must decide that the phrase means "bad". This is why negation,
context, and domain language are recurring challenges.

## 12. Practice area 7: concept extraction

Concept extraction groups words and phrases into semantically similar sets. A simple statistical
version is to rank terms by TF-IDF within each category and inspect the top terms.

In [11]:
category_terms = {}

for category in sorted(df["category"].unique()):
    mask = (df["category"] == category).to_numpy()
    mean_tfidf = np.asarray(X[mask].mean(axis=0)).ravel()
    top_indices = np.argsort(mean_tfidf)[-6:][::-1]
    category_terms[category] = terms[top_indices]

concept_table = pd.DataFrame(dict(category_terms))
concept_table

,account,billing,technical
0,workspace,refund,app
1,profile,invoice,error
2,team,twice,update
3,update,subscription,log
4,share workspace,charge,message
5,workspace team,invoice refund,shows error


The top terms are interpretable but shallow: they reflect co-occurrence, not a true ontology. More
advanced concept extraction uses embeddings, synonym dictionaries, taxonomies, and human review.

## 13. Put it together: a text analytics workflow

A repeatable workflow keeps the project honest:

1. **Define the decision.** What action will the analysis support?
2. **Acquire and document the corpus.** Source, time span, licence, privacy, sampling.
3. **Preprocess and represent.** Cleaning, tokenisation, TF-IDF or embeddings.
4. **Choose a practice area.** Search, clustering, classification, web mining, extraction, NLP, or
   concept extraction.
5. **Model and baseline.** Start simple; compare against a trivial baseline.
6. **Evaluate.** Use task-specific metrics, error analysis, and a held-out set.
7. **Deploy and monitor.** Drift, feedback loops, privacy, and human oversight.

The workflow is iterative. A bad evaluation often sends you back to problem framing or data
collection, not to a more complex model.

## 14. Summarisation

Summarisation compresses a document or collection while preserving the important information.

- **Extractive summarisation** selects existing sentences or phrases.
- **Abstractive summarisation** generates new wording.
- **Term extraction** lists representative terms.
- **Sentence extraction** selects representative sentences.

A simple extractive baseline scores each sentence by cosine similarity to the document centroid,
then returns the top sentences in their original order. This is a centroid-based method, related to
the classic vector-space approach.

In [12]:
article = (
    "The refund policy allows customers to request a refund within 30 days of purchase. "
    "Refunds are returned to the original payment method within five to ten business days. "
    "Shipping charges are not refundable unless the item arrived damaged. "
    "To start a refund, open the order history page and select the request refund option. "
    "Customers must include the order number and a short reason for the return. "
    "Damaged items require a photo before the support team can approve the request. "
    "Digital products are refundable only if they have not been downloaded. "
    "The support team reviews every request and sends a decision by email."
)

sentences = [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+", article) if sentence.strip()]
print("Number of sentences:", len(sentences))
for index, sentence in enumerate(sentences, start=1):
    print(f"{index}. {sentence}")

Number of sentences: 8
1. The refund policy allows customers to request a refund within 30 days of purchase.
2. Refunds are returned to the original payment method within five to ten business days.
3. Shipping charges are not refundable unless the item arrived damaged.
4. To start a refund, open the order history page and select the request refund option.
5. Customers must include the order number and a short reason for the return.
6. Damaged items require a photo before the support team can approve the request.
7. Digital products are refundable only if they have not been downloaded.
8. The support team reviews every request and sends a decision by email.


In [13]:
sentence_vectorizer = TfidfVectorizer(stop_words="english")
sentence_matrix = sentence_vectorizer.fit_transform(sentences)
centroid = np.asarray(sentence_matrix.mean(axis=0))
sentence_scores = cosine_similarity(sentence_matrix, centroid).ravel()

score_table = pd.DataFrame(
    {
        "sentence": sentences,
        "centroid_similarity": sentence_scores,
    }
).sort_values("centroid_similarity", ascending=False)

score_table.round(3)

,sentence,centroid_similarity
0,The refund policy allows customers to request ...,0.501
3,"To start a refund, open the order history page...",0.469
5,Damaged items require a photo before the suppo...,0.459
7,The support team reviews every request and sen...,0.428
2,Shipping charges are not refundable unless the...,0.382
4,Customers must include the order number and a ...,0.360
6,Digital products are refundable only if they h...,0.349
1,Refunds are returned to the original payment m...,0.333


In [14]:
n_sentences = 3
selected = np.argsort(sentence_scores)[::-1][:n_sentences]
summary = " ".join(sentences[index] for index in sorted(selected))

print("Extractive summary:")
print(summary)

Extractive summary:
The refund policy allows customers to request a refund within 30 days of purchase. To start a refund, open the order history page and select the request refund option. Damaged items require a photo before the support team can approve the request.


The summary is faithful because it uses original sentences, but it may be redundant or miss
context. A production summariser would add redundancy penalties, position features, query focus, and
a proper evaluation such as ROUGE. Abstractive summarisers can be more fluent but may hallucinate.
Always check factual consistency.

## 15. Evaluation and cautions

Text analytics is easy to demo and hard to evaluate. Common failure modes:

- **Leakage.** Fitting the vectorizer on the full corpus before splitting.
- **Sampling bias.** The corpus does not represent the population you care about.
- **Label noise.** Human annotations disagree or encode inconsistent definitions.
- **Class imbalance.** Accuracy hides poor performance on rare but important classes.
- **Domain drift.** Vocabulary and intent change over time.
- **Privacy and copyright.** Text often contains personal or licensed content.
- **Spurious correlation.** A keyword may correlate with a label without being a reliable signal.
- **Overclaiming semantics.** A topic model or sentiment score is not ground truth.
- **Summarisation errors.** Extractive summaries can omit critical context; abstractive summaries
  can invent facts.

Evaluate with task-specific metrics, a held-out set, and human review of errors. For search, use
precision@k, recall, or nDCG. For classification, use per-class precision and recall. For
summarisation, use ROUGE plus a factual-consistency check.

## Summary

- Text analytics combines IR, IE, NLP, statistics, ML, and web mining.
- The seven practice areas are search/IR, clustering, classification, web mining, information
  extraction, NLP, and concept extraction.
- The generalized vector-space model and TF-IDF remain useful baselines.
- A good workflow starts with a decision and a documented corpus, not with a model.
- Extractive summarisation is a strong, faithful baseline.
- Evaluation, privacy, and human oversight are part of the method, not an afterthought.

## Exercises

1. Replace TF-IDF with count vectors and compare search rankings and classification accuracy.
2. Add a fourth support category and see how clustering and classification respond.
3. Implement precision@k for the search query and compare two weighting schemes.
4. Extend the information-extraction patterns to capture product names or subscription plans.
5. Build a bigram concept table and identify phrases that unigrams miss.
6. Add a redundancy penalty to the summariser and compare the selected sentences.
7. Write a one-page project brief for a text-analytics problem, naming the practice area, data,
   baseline, metric, and deployment risk.